In [1]:
import pandas as pd

In [2]:
df = pd.read_csv(r"dataset\BBC News Train.csv")
print(f'''
      Shape: {df.shape}
      Columns: {df.columns.tolist()}
      Category column values: {df['Category'].unique().tolist()}
''')


      Shape: (1490, 3)
      Columns: ['ArticleId', 'Text', 'Category']
      Category column values: ['business', 'tech', 'politics', 'sport', 'entertainment']



## Data filtering

In [3]:
tech_data = df[df.Category=='tech']
print(tech_data.shape)
tech_data.head()

(261, 3)


,ArticleId,Text,Category
3,1976,lifestyle governs mobile choice faster bett...,tech
19,1552,moving mobile improves golf swing a mobile pho...,tech
24,405,bt boosts its broadband packages british telec...,tech
26,702,peer-to-peer nets here to stay peer-to-peer ...,tech
30,1951,pompeii gets digital make-over the old-fashion...,tech


In [4]:
import nltk
nltk.download('punkt')
from nltk.tokenize import sent_tokenize

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\gaura\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [5]:
tech_data = tech_data.copy()

tech_data["num_sent"] = tech_data["Text"].apply(lambda x: len(sent_tokenize(x)))
filtered_data = tech_data[tech_data['Text'].str.contains('microsoft',case=False,na=False)]

filtered_data = filtered_data.copy()
filtered_data = filtered_data[(filtered_data["num_sent"] <= 15)]
filtered_data = filtered_data.drop_duplicates(subset=['Text'], keep='first')

filtered_data.to_csv(r'filtered_data.csv',index=False)

In [6]:
print(filtered_data.shape)
filtered_data.sort_values('num_sent')

(9, 4)


,ArticleId,Text,Category,num_sent
301,428,microsoft takes on desktop search microsoft ha...,tech,10
651,1615,microsoft seeking spyware trojan microsoft is ...,tech,10
428,69,microsoft gets the blogging bug software giant...,tech,13
509,2042,virus poses as christmas e-mail security firms...,tech,13
861,184,xbox power cable fire fear microsoft has sai...,tech,13
1179,728,anti-spam laws bite spammer hard the net s sel...,tech,13
48,277,halo 2 sells five million copies microsoft is ...,tech,14
589,1604,joke e-mail virus tricks users a virus that di...,tech,15
1305,1004,microsoft releases bumper patches microsoft ha...,tech,15


## Sentence Spliting

In [7]:
sentences = []
source_ids = []

for idx, row in filtered_data.iterrows():
    article_id = row["ArticleId"]
    text = row["Text"]
    
    sents = sent_tokenize(text)
    
    for s in sents:
        s = s.strip()
        if len(s) > 0:
            sentences.append(s)
            source_ids.append(article_id)

In [8]:
sent_df = pd.DataFrame({
    "article_id": source_ids,
    "sentence": sentences
})

In [9]:
print(sent_df.shape)
sent_df.head(20)

(116, 2)


,article_id,sentence
0,277,halo 2 sells five million copies microsoft is ...
1,277,halo 2 has proved popular online with gamers ...
2,277,according to microsoft nine out of 10 xbox li...
3,277,the sequel to the best-selling need for speed:...
4,277,the racing game moved up one spot to first pla...
5,277,halo 2 dropped one place to five while half-l...
6,277,last week s new releases goldeneye: rogue age...
7,277,record numbers of warcraft fans are settling i...
8,277,on the opening day of the world of warcraft ma...
9,277,on the evening of the first day more than 100 ...


## Sentence preprocessing

In [10]:
import re
import unicodedata

In [11]:
clean_sentences = []

for s in sent_df["sentence"]:
    original = s
    
    s = s.strip()
    
    if len(s.split()) < 5:
        continue
    
    if not re.search(r"[A-Za-z]", s):
        continue
    
    if len(re.sub(r"[A-Za-z0-9]", "", s)) / len(s) > 0.5:
        continue
    
    bad_patterns = [
        r"click here", r"share this", r"subscribe", r"read more",
        r"copyright", r"all rights reserved"
    ]
    if any(re.search(pat, s.lower()) for pat in bad_patterns):
        continue
    
    clean_sentences.append(s)

In [12]:

filtered_sent_df = pd.DataFrame({
    "article_id": source_ids,
    "sentence": clean_sentences
    })

print("Before:", sent_df.shape)
print("After:", filtered_sent_df.shape)

Before: (116, 2)
After: (116, 2)


In [13]:
filtered_sent_df.head()

,article_id,sentence
0,277,halo 2 sells five million copies microsoft is ...
1,277,halo 2 has proved popular online with gamers ...
2,277,according to microsoft nine out of 10 xbox li...
3,277,the sequel to the best-selling need for speed:...
4,277,the racing game moved up one spot to first pla...


In [14]:
def light_clean(text):
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'[-_]{2,}', ' ', text)
    text = re.sub(r'([!?.,]){2,}', r'\1', text)
    text = ''.join(ch for ch in text if unicodedata.category(ch)[0] != "C")
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = text.lower()
    
    return text

In [15]:
df_sent = filtered_sent_df.copy()

df_sent["clean_sentence"] = df_sent["sentence"].astype(str).apply(light_clean)

In [16]:
before = df_sent.shape[0]
df_sent = df_sent[df_sent["clean_sentence"].str.strip() != ""].reset_index(drop=True)
after = df_sent.shape[0]

final_df = df_sent[["article_id", "clean_sentence"]].rename(columns={"clean_sentence": "sentence"})

In [17]:
print(f"Sentences before cleaning: {before}")
print(f"Sentences after cleaning & removing empties: {after}")

Sentences before cleaning: 116
Sentences after cleaning & removing empties: 116


In [18]:
final_df.head()

,article_id,sentence
0,277,halo 2 sells five million copies microsoft is ...
1,277,halo 2 has proved popular online with gamers n...
2,277,according to microsoft nine out of 10 xbox liv...
3,277,the sequel to the best-selling need for speed:...
4,277,the racing game moved up one spot to first pla...


## Sentence Embedings

In [19]:
from sentence_transformers import SentenceTransformer
import numpy as np

In [21]:
MODEL_NAME = "all-MiniLM-L6-v2"
BATCH_SIZE = 64 

model = SentenceTransformer(MODEL_NAME)

In [22]:
sentences = final_df["sentence"].astype(str).tolist()
print("Number of sentences to encode:", len(sentences))

Number of sentences to encode: 116


In [24]:
embeddings = model.encode(sentences,
                batch_size=BATCH_SIZE,
                show_progress_bar=True,
                convert_to_numpy=True
            )

print("Embeddings shape:", embeddings.shape)

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Embeddings shape: (116, 384)


In [28]:
sent_emb_df = final_df.reset_index(drop=True).copy()
sent_emb_df["emb_index"] = np.arange(len(sent_emb_df))

sent_emb_df.head()

,article_id,sentence,emb_index
0,277,halo 2 sells five million copies microsoft is ...,0
1,277,halo 2 has proved popular online with gamers n...,1
2,277,according to microsoft nine out of 10 xbox liv...,2
3,277,the sequel to the best-selling need for speed:...,3
4,277,the racing game moved up one spot to first pla...,4


## Dimensionality reduction

In [29]:
import umap

In [32]:
print("Original embedding shape:", embeddings.shape)

Original embedding shape: (116, 384)


In [35]:
n_components_cluster = 20   # You can adjust between 10 and 30
n_neighbors = 15            # Controls how local/global the structure is

In [36]:
umap_cluster = umap.UMAP(
    n_components=n_components_cluster,
    n_neighbors=n_neighbors,
    metric='cosine',
    random_state=42
)

emb_reduced = umap_cluster.fit_transform(embeddings)
print("Reduced embedding shape:", emb_reduced.shape)

c:\Users\gaura\AppData\Local\Programs\Python\Python311\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Reduced embedding shape: (116, 20)


In [37]:
umap_2d = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    metric='cosine',
    random_state=42
)

emb_2d = umap_2d.fit_transform(embeddings)
print("2D embedding shape:", emb_2d.shape)

c:\Users\gaura\AppData\Local\Programs\Python\Python311\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


2D embedding shape: (116, 2)


In [38]:
sent_emb_df["umap_x"] = emb_2d[:, 0]
sent_emb_df["umap_y"] = emb_2d[:, 1]

sent_emb_df.head()

,article_id,sentence,emb_index,umap_x,umap_y
0,277,halo 2 sells five million copies microsoft is ...,0,3.419658,5.107649
1,277,halo 2 has proved popular online with gamers n...,1,3.447265,5.453901
2,277,according to microsoft nine out of 10 xbox liv...,2,3.672795,5.414927
3,277,the sequel to the best-selling need for speed:...,3,2.938107,5.294118
4,277,the racing game moved up one spot to first pla...,4,3.241683,4.967222
